# Assignment 3: Fine-tuning language models

In this assignment, you will perform supervised fine-tuning (SFT) of a small open LLM on an instruction tuning dataset. You will convert this dataset into instruction-response pairs, fine-tune a causal language model using LoRA (Low-Rank Adaptation), and evaluate it through prompted inference and comparison with other methods.

## Preliminaries

First, let's install the required libraries. If you are running in your own environment, make sure the following are installed:

- [Torch](https://docs.pytorch.org/docs/stable/index.html)
- [Transformers](https://huggingface.co/docs/transformers/index)
- [Datasets](https://huggingface.co/docs/datasets/index)
- [Evaluate](https://huggingface.co/docs/evaluate/en/index)
- [NLTK](https://www.nltk.org/api/nltk.html)
- [rouge_score](https://pypi.org/project/rouge-score/)

In a Colab notebook, most of them are already installed, except Evaluate and rouge_score.

In [8]:
%pip install evaluate rouge_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.5 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=0edd0af0da775a0441737a5bde1766f2a5e9957c405afb19476f8f18d6a744ab
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


We also set some configuration parameters.

Most importantly, you should select a language model to work with in this assignment and enter its HuggingFace identifier in the parameter `MODEL_NAME` below. In principle you can use any model that you want, but we recommend that you select a model that has not already been trained to follow instructions, so it should be a "pure" language model trained on raw text (similar to Assignments 1 and 2).

The selected model should be small enough to fit in your computational environment. We have verified that the 135-million parameter [`SmolLM2` model](https://huggingface.co/HuggingFaceTB/SmolLM2-135M), developed by HuggingFace, can be used to solve this assignment in a Colab notebook (free tier, T4 GPU). If you run on a cluster, you can select a larger model (and probably see more interesting results).

We also define training and test set sizes here. Again, the values below have been set so that the assignment can be solved in Colab, and you can increase these sizes to improve the quality of the fine-tuned models.

In [9]:
import torch
SEED = 101
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MAX_TRAIN_SAMPLES = 5000
MAX_TEST_SAMPLES = 400

MODEL_NAME = "HuggingFaceTB/SmolLM2-135M"

# Part 1: Preprocessing

### ⚙&nbsp; Task 1.1: Loading and inspecting the dataset

The dataset [SmolTalk](https://huggingface.co/datasets/HuggingFaceTB/smoltalk) is a collection of instruction-response pairs designed for SFT of large language models for instruction following. This dataset consists of examples of user inputs with system responses.

You can load using the datasets from the HuggingFace repository as follows.

In [10]:
from datasets import load_dataset
from datasets import DatasetDict

smoltalk = load_dataset("HuggingFaceTB/smoltalk", 'all')

README.md:   0%|          | 0.00/9.72k [00:00<?, ?B/s]

data/all/train-00000-of-00009.parquet:   0%|          | 0.00/223M [00:00<?, ?B/s]

data/all/train-00001-of-00009.parquet:   0%|          | 0.00/223M [00:00<?, ?B/s]

data/all/train-00002-of-00009.parquet:   0%|          | 0.00/223M [00:00<?, ?B/s]

data/all/train-00003-of-00009.parquet:   0%|          | 0.00/223M [00:00<?, ?B/s]

...

data/all/train-00007-of-00009.parquet:   0%|          | 0.00/224M [00:00<?, ?B/s]

data/all/train-00008-of-00009.parquet:   0%|          | 0.00/224M [00:00<?, ?B/s]

data/all/test-00000-of-00001.parquet:   0%|          | 0.00/105M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1043917 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/54948 [00:00<?, ? examples/s]

In order to make this assignment possible to solve in a restricted environment, we simplify the dataset a bit:
- We remove multi-turn chat dialogues from the dataset;
- We remove instances where the query or the answer is greater than a set maximum length;
- We keep a subset of the data for training and testing (by default 5000 and 400, respectively).

In [11]:

# Filter 1: Keep conversations with 3 or fewer messages (remove long multi-turn chats)
# Filter 2: Ensure no single message exceeds 256 characters to save memory during tokenization
smoltalk_simplified = smoltalk.filter(lambda row: len(row['messages']) <= 3 and all(len(m['content']) <= 256 for m in row['messages']))
smoltalk_simplified = DatasetDict({
    "train": smoltalk_simplified["train"].select(range(MAX_TRAIN_SAMPLES)),
    "test": smoltalk_simplified["test"].select(range(MAX_TEST_SAMPLES)),
})

Filter:   0%|          | 0/1043917 [00:00<?, ? examples/s]

Filter:   0%|          | 0/54948 [00:00<?, ? examples/s]

In [ ]:

smoltalk_simplified = smoltalk.filter(lambda row: len(row['messages']) <= 3 and all(len(m['content']) <= 256 for m in row['messages']))

# Truncate the dataset to our predefined maximum samples for fast iteration
smoltalk_simplified = DatasetDict({
    "train": smoltalk_simplified["train"].select(range(MAX_TRAIN_SAMPLES)),
    "test": smoltalk_simplified["test"].select(range(MAX_TEST_SAMPLES)),
})

In [12]:
smoltalk_simplified

DatasetDict({
    train: Dataset({
        features: ['messages', 'source'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['messages', 'source'],
        num_rows: 400
    })
})

Print some examples from the dataset so that we understand the format.

Each example from the training or test set consists of a sequence of messages. The number of messages in each example will be 2 or 3, because we removed multi-turn chat dialogues in the previous step. Each message is associated with a `role` label:
- `user`: an example of something the user might write.
- `assistant`: an example of an output an LLM could be expected to produce, given the input.
- `system`: a *system prompt* that gives guidelines for the general behavior of the LLM's behavior.

All examples in the dataset include a user input and an assistant output, but the system prompt is not available in all of the examples.

In [13]:
smoltalk_simplified['train'][0]

{'messages': [{'content': "You are an AI rewriting assistant. You will be provided with a text and you need to rewrite it according to the user's instructions.",
   'role': 'system'},
  {'content': 'Rearrange this sentence to make it easy to understand:\nThe restaurant ran out of food, so the chef made some more.',
   'role': 'user'},
  {'content': 'The chef made more food after the restaurant ran out.',
   'role': 'assistant'}],
 'source': 'explore-instruct-rewriting'}

### 🎓&nbsp; Task 1.2: Formatting the data for instruction tuning

Define a function `format_input_output` that converts an example from the dataset into an input/output pair that we can use to fine-tune the LLM.

You are free to design the format. The following document gives some examples that have been used by different instruction-following LLMs including Llama and Mistral: https://huggingface.co/learn/llm-course/chapter11/2#common-template-formats

The later stages of our preprocessing pipeline expect that this function returns an object containing two parts: the `prompt` (what goes into the LLM before generating anything) and the `response` (what the LLM is expected to generate).

In [14]:
def format_input_output(example):
    """
    Converts dataset messages into a ChatML-like prompt/response format.
    This structural formatting helps the model differentiate roles during SFT.
    Language models only understand continuous strings of text.
    By injecting explicit control tokens (like <|im_start|> and <|im_end|>), we teach the model to recognize "who is speaking" and "when to stop speaking".
    """
    messages = example['messages']
    prompt = ""
    response = ""

    # Iterate through all messages except the final one to build the context prompt
    for msg in messages[:-1]:
        if msg['role'] == 'system':
            prompt += f"<|im_start|>system\n{msg['content']}<|im_end|>\n"
        elif msg['role'] == 'user':
            prompt += f"<|im_start|>user\n{msg['content']}<|im_end|>\n"
        elif msg['role'] == 'assistant':
            prompt += f"<|im_start|>assistant\n{msg['content']}<|im_end|>\n"

    # Handle the final message to set up the generative conditioning
    last_msg = messages[-1]
    if last_msg['role'] == 'user':

        prompt += f"<|im_start|>user\n{last_msg['content']}<|im_end|>\n"
        prompt += "<|im_start|>assistant\n"
    else:
        # If the last message is the assistant's response (common in simplified datasets)
        response = f"{last_msg['content']}<|im_end|>"
        # Find the preceding user message to ensure the prompt is properly closed
        for msg in messages:
            if msg['role'] == 'user':
                prompt += f"<|im_start|>user\n{msg['content']}<|im_end|>\n"
        prompt += "<|im_start|>assistant\n"

    return {"prompt": prompt, "response": response}

In [ ]:
def format_input_output(example):
    """
    Converts raw JSON dataset messages into a ChatML-like prompt/response text format.

    Why ChatML?
    Language models only understand continuous strings of text. By injecting explicit
    control tokens (like <|im_start|> and <|im_end|>), we teach the model to recognize
    "who is speaking" and "when to stop speaking".
    """
    messages = example['messages']
    prompt = ""
    response = ""

    # Step 1: Build the 'Context Prompt' (Everything except the final assistant answer)
    # We iterate through all messages EXCEPT the very last one.
    for msg in messages[:-1]:
        if msg['role'] == 'system':
            prompt += f"<|im_start|>system\n{msg['content']}<|im_end|>\n"
        elif msg['role'] == 'user':
            prompt += f"<|im_start|>user\n{msg['content']}<|im_end|>\n"
        elif msg['role'] == 'assistant':
            prompt += f"<|im_start|>assistant\n{msg['content']}<|im_end|>\n"

    # Step 2: Handle the final message to create the generative target
    last_msg = messages[-1]

    if last_msg['role'] == 'user':
        # Inference Scenario: The user asked something, but we have no answer yet.
        # We append the user query, and then open the assistant tag without closing it.
        # This forces the model to "fill in the blank" starting as the assistant.
        prompt += f"<|im_start|>user\n{last_msg['content']}<|im_end|>\n"
        prompt += "<|im_start|>assistant\n"
    else:
        # Training Scenario: The last message is the Assistant's ground-truth answer.
        # We append <|im_end|> to the response so the model learns to output the stop signal.
        response = f"{last_msg['content']}<|im_end|>"

        # We still need to "prime" the prompt to trigger the assistant's turn.
        # Note: If messages[:-1] already added the preceding user message, we just open the tag.
        prompt += "<|im_start|>assistant\n"

    return {"prompt": prompt, "response": response}

Apply the function you implemented to the dataset as a whole.

In [15]:
ds_sft = smoltalk_simplified.map(format_input_output)

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Then verify that the dataset now contains the new fields you created.

In [16]:
ds_sft['train'][0]

{'messages': [{'content': "You are an AI rewriting assistant. You will be provided with a text and you need to rewrite it according to the user's instructions.",
   'role': 'system'},
  {'content': 'Rearrange this sentence to make it easy to understand:\nThe restaurant ran out of food, so the chef made some more.',
   'role': 'user'},
  {'content': 'The chef made more food after the restaurant ran out.',
   'role': 'assistant'}],
 'source': 'explore-instruct-rewriting',
 'prompt': "<|im_start|>system\nYou are an AI rewriting assistant. You will be provided with a text and you need to rewrite it according to the user's instructions.<|im_end|>\n<|im_start|>user\nRearrange this sentence to make it easy to understand:\nThe restaurant ran out of food, so the chef made some more.<|im_end|>\n<|im_start|>user\nRearrange this sentence to make it easy to understand:\nThe restaurant ran out of food, so the chef made some more.<|im_end|>\n<|im_start|>assistant\n",
 'response': 'The chef made more 

### ⚙&nbsp; Task 1.3: Tokenizing the dataset

We will now prepare the format required by the HuggingFace Trainer.

We first load the tokenizer for our selected model:

In [17]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

tokenizer_config.json:   0%|          | 0.00/3.66k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/831 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

Write a function `tokenize_helper` that takes an example (using the prompt/response format from the previous step) and produces the following three results:

- `input_ids`: the integer token ids of the concatenated prompt and response;
- `labels`: a list of the same length as `input_ids`, where the response token ids are the same, but where the prompt token ids have all been replaced by the loss masking identifier -100.
- `attention_mask`: the attention mask. This should just be a list of the same length as the other two lists, with all items set to 1.

The reason why `input_ids` and `labels` are different is that
we do not want to compute the training loss for tokens that appear in the user's input. We want to train the model to generate output *conditionally*: based on a prompt. But why the magic number -100? This is the number used by default in PyTorch's [`CrossEntropyLoss`](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html) to indicate an item that should be excluded in loss computations. (This issue was also mentioned in [Assignment 1](https://liu-nlp.ai/dl4nlp/units/a1_1.html#task-4.1-implementing-the-trainer).)

In [18]:
def tokenize_helper(example):
    prompt = example['prompt']
    response = example['response']

    # Tokenize separately to calculate the exact length of the prompt
    prompt_tokens = tokenizer(prompt, add_special_tokens=False)
    response_tokens = tokenizer(response, add_special_tokens=False)

    # Concatenate inputs and masks
    input_ids = prompt_tokens['input_ids'] + response_tokens['input_ids']
    attention_mask = prompt_tokens['attention_mask'] + response_tokens['attention_mask']

    # Mask out the prompt tokens using -100 so they do not contribute to the loss computation
    labels = [-100] * len(prompt_tokens['input_ids']) + response_tokens['input_ids']

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }

As above, apply the function you implemented to the dataset using `map`. This will add the three new fields to the dataset.

In [19]:
tokenized_ds_sft = ds_sft.map(tokenize_helper)

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]


## Part 2: Evaluation of the baseline model

As a first step, we will see how well the *baseline* model performs: that is, a model that has not been trained to follow instructions.

### ⚙&nbsp; Task 2.1: Preparing for evaluation

In this section, we set up a few utilities we will need to complete our training and evaluation infrastructure. These utilities will be given and you don't need to modify anything.

The first piece we need is a *collator*: that is, a tool that takes a number of instances and creates PyTorch tensors for a training batch. To make the batch fit into rectangular tensors, padding tokens will be added.

In [20]:
def data_collator(batch):
    """
    Create a custom collate function for causal language modeling.

    Args:
        batch: List of examples, each with 'input_ids', 'attention_mask', 'labels'
        tokenizer: Tokenizer with pad_token_id
    """

    input_ids_list = [torch.tensor(example["input_ids"], dtype=torch.long) for example in batch]
    attention_masks_list = [torch.tensor(example["attention_mask"], dtype=torch.long) for example in batch]
    labels_list = [torch.tensor(example['labels'], dtype=torch.long) for example in batch]

    # Find max length in this batch
    max_len = max(x.size(0) for x in input_ids_list)

    # Helper pad function
    def pad_to_max(x_list, pad_value):
        padded = []
        for x in x_list:
            pad_len = max_len - x.size(0)
            if pad_len > 0:
                pad_tensor = torch.full((pad_len,), pad_value, dtype=x.dtype)
                x = torch.cat([x, pad_tensor], dim=0)
            padded.append(x)
        return torch.stack(padded, dim=0)

    # Use tokenizer.pad_token_id for inputs, 0 for attention_mask, -100 for labels
    pad_id = tokenizer.pad_token_id

    batch_input_ids = pad_to_max(input_ids_list, pad_value=pad_id)
    batch_attention_mask = pad_to_max(attention_masks_list, pad_value=0)
    batch_labels = pad_to_max(labels_list, pad_value=-100)

    batch = {
            "input_ids": batch_input_ids,
            "attention_mask": batch_attention_mask,
            "labels": batch_labels,
        }
    return batch

The second utility we need is an evaluator. We will use the **ROUGE-L** metric, which computes the longest common subsequence between the model's output and the gold-standard answer. You can read about ROUGE-L here: https://en.wikipedia.org/wiki/ROUGE_(metric)

When using the ROUGE-L metric in a Trainer, we need to wrap it in an object defined as follows:

In [21]:
import evaluate

class RougeMetricComputer:
    """
    Stateful metric for batch_eval_metrics=True.

    It:
      - accumulates predictions and references across batches
      - computes ROUGE-L once at the end (compute_result=True)
    """

    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
        self.rouge = evaluate.load("rouge")
        self.all_predictions = []
        self.all_references = []

    def __call__(self, eval_pred, compute_result=False):
        """Accumulate predictions and compute at the end."""

        logits, labels = eval_pred
        pred_ids = logits.argmax(axis=-1)

        # Collect decoded answer-span text from each example in the batch
        for p, lbl in zip(pred_ids, labels):
            mask = lbl != -100
            if mask.sum() == 0:
                continue

            ref_ids = lbl[mask]
            pred_ids_filtered = p[mask]

            ref_text = self.tokenizer.decode(ref_ids, skip_special_tokens=True)
            pred_text = self.tokenizer.decode(
                pred_ids_filtered, skip_special_tokens=True,
                eos_token_id=self.tokenizer.vocab.get('<|im_end|>', self.tokenizer.eos_token_id)
            )

            self.all_references.append(ref_text.strip())
            self.all_predictions.append(pred_text.strip())

        # Only compute at the very end of eval
        if compute_result:
            if len(self.all_references) > 0:
                scores = self.rouge.compute(
                    predictions=self.all_predictions,
                    references=self.all_references,
                )

                # Clear accumulated data for next eval call
                self.all_predictions = []
                self.all_references = []
                return {"rougeL": scores["rougeL"]}
            else:
                return {}
        else:
            return {}

compute_metrics = RougeMetricComputer(tokenizer)


Finally, we make a function that sets up a [`Trainer`](https://huggingface.co/docs/transformers/main_classes/trainer).

In [22]:
from transformers import Trainer
from transformers.trainer_callback import ProgressCallback

def make_trainer(model, training_args):
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_ds_sft["train"],
        eval_dataset=tokenized_ds_sft["test"],
        compute_metrics=compute_metrics,
        data_collator=data_collator,
    )
    trainer.callback_handler.callbacks = [
        cb for cb in trainer.callback_handler.callbacks
        if type(cb).__name__ != "NotebookProgressCallback"
    ]
    trainer.add_callback(ProgressCallback)
    return trainer


### 🎓&nbsp; Task 2.2: Evaluating the pre-trained model

Now, we have all the pieces to evaluate our baseline model that has not been instruction-tuned.

The following code will compute the loss on the test set as well as the ROUGE-L score. You will later compare these scores to the models that you train.

Why do you think the ROUGE-L score is as high as it is, even without any training for instruction-following?

Note:
ROUGE-L calculates the Longest Common Subsequence (LCS) between the generated text and the reference answer.
The "Stopword" Padding: Pre-trained models naturally generate grammatically correct English.  Therefore, they will produce many common functional words (e.g., "the", "is", "and", "to") in roughly the same structural order as the reference text, which artificially inflates the LCS.
Prompt Echoing: Base models often "echo" or continue the phrasing of the user's prompt.  If the user asks, "What are the stages of mitosis?", the model might start generating, "The stages of mitosis are...", which heavily overlaps with a human reference answer that uses the same introductory phrasing.

In [23]:
from transformers import TrainingArguments
from transformers import AutoModelForCausalLM
import time
import json

print("\n" + "=" * 80)
print("EVALUATING PRETRAINED MODEL (BASELINE)")
print("=" * 80)

# Load pure base model weights directly to GPU
pretrained_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to("cuda")

# Set up evaluation configuration
pretrained_eval_args = TrainingArguments(
    eval_strategy="no",              # Disable periodic evaluation during training (since we aren't training)
    per_device_eval_batch_size=1,    # Process 1 sample at a time to minimize GPU VRAM usage
    bf16=True, fp16=False,           # Use bfloat16 mixed precision for faster math and half memory consumption
    report_to="none",                # Don't send logs to external trackers like wandb
    batch_eval_metrics=True,         # Enable accumulating metrics per batch
    eval_accumulation_steps=1,       # CRITICAL for memory: Offload prediction tensors to CPU every 1 step to prevent OOM
)

pretrained_trainer = make_trainer(pretrained_model, pretrained_eval_args)

t0 = time.perf_counter()
pretrained_eval_metrics = pretrained_trainer.evaluate()
pretrained_eval_time = time.perf_counter() - t0

pretrained_eval_loss = float(pretrained_eval_metrics["eval_loss"])
pretrained_rougeL = pretrained_eval_metrics.get("eval_rougeL", None)

print("\nPRETRAINED EVAL METRICS:")
print(json.dumps(pretrained_eval_metrics, indent=2))


EVALUATING PRETRAINED MODEL


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

  0%|          | 0/400 [00:00<?, ?it/s]


PRETRAINED EVAL METRICS:
{
  "eval_loss": 2.595301628112793,
  "eval_model_preparation_time": 0.0041,
  "eval_rougeL": 0.5822138058898996,
  "eval_runtime": 34.7975,
  "eval_samples_per_second": 11.495,
  "eval_steps_per_second": 11.495,
  "epoch": 0
}



## Part 3: Supervised fine-tuning



### 🎓&nbsp; Task 3.1: Training the full model

Next, we train the pre-trained model using SFT over all the parameters, then calculate the metrics and outputs to evaluate how well it follows instructions.

How do the results differ from those in the previous step?

In [24]:
# SFT Training Configuration
baseline_training_args = TrainingArguments(
    eval_strategy="epoch",           # Evaluate performance at the end of every epoch
    logging_steps=2000,              # Print training loss to console every 2000 steps
    save_strategy="no",              # Don't save intermediate checkpoint weights (saves disk space on Colab)
    num_train_epochs=1,              # For this assignment, 1 pass over the dataset is sufficient
    per_device_train_batch_size=1,   # Keep training batch size small to avoid OOM
    per_device_eval_batch_size=1,
    bf16=True, fp16=False,           # Enable mixed precision training
    report_to="none",
    batch_eval_metrics=True,
    eval_accumulation_steps=1,
)

base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to("cuda")
baseline_trainer = make_trainer(base_model, baseline_training_args)

print("\n" + "=" * 80)
print("TRAINING FULL SFT MODEL")
print("=" * 80)
# Start full-parameter backpropagation (this will update all 135M parameters)
baseline_trainer.train()

print("\n" + "=" * 80)
print("EVALUATING FULL SFT MODEL")
print("=" * 80)
baseline_eval_metrics = baseline_trainer.evaluate()
print("\nSFT EVAL METRICS:")
print(json.dumps(baseline_eval_metrics, indent=2))

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]


TRAINING FULL SFT MODEL


  0%|          | 0/5000 [00:00<?, ?it/s]

{'loss': '1.687', 'grad_norm': '4.281', 'learning_rate': '3.001e-05', 'epoch': '0.4'}
{'loss': '1.479', 'grad_norm': '9.812', 'learning_rate': '1.001e-05', 'epoch': '0.8'}


  0%|          | 0/400 [00:00<?, ?it/s]

{'eval_loss': '1.512', 'eval_rougeL': '0.6371', 'eval_runtime': '32.13', 'eval_samples_per_second': '12.45', 'eval_steps_per_second': '12.45', 'epoch': '1'}
{'train_runtime': '765.5', 'train_samples_per_second': '6.532', 'train_steps_per_second': '6.532', 'train_loss': '1.547', 'epoch': '1'}

EVALUATING FULL SFT MODEL


  0%|          | 0/400 [00:00<?, ?it/s]


SFT EVAL METRICS:
{
  "eval_loss": 1.5118478536605835,
  "eval_rougeL": 0.6370613398338236,
  "eval_runtime": 32.8589,
  "eval_samples_per_second": 12.173,
  "eval_steps_per_second": 12.173,
  "epoch": 1.0
}


### ⚙&nbsp; Task 3.3: Counting the number of trainable parameters

Define a function `num_trainable_parameters` that computes the number of floating-point numbers that a given model will update during training.

**Hints**:
- For a PyTorch module `m`, you can use `m.parameters()` to access its parameter tensors.
- However, you should only include parameter tensors where the flag `requires_grad` is True.


In [25]:
def num_trainable_parameters(model):
    """Count number of trainable parameters.

    Args:
        model: A PyTorch module.
    """
    # Sum the number of elements (numel) for each parameter tensor that requires gradients
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Trainable parameters in base SFT model: {num_trainable_parameters(base_model)}")

Trainable parameters in base SFT model: 134515008


## Part 4: Parameter-efficient fine-tuning

In the last section of this assignment, we will use LoRA to train the model in a more parameter-efficient manner. You may want to prepare by reading  by [the paper by Hu et al. (2021)](https://arxiv.org/pdf/2106.09685) and the teaching material provided for this course.

### ⚙&nbsp; Task 4.1: Utilities for modifying models

Define a function `extract_lora_targets` that extracts the relevant linear layers from all Transformer blocks in your selected LLM.
It is up to you to decide what layers to select; in the experiments described in the original LoRA paper, the query and value projection matrices were fine-tuned with LoRA, while all other layers were left unchanged.
Return a dictionary that maps the component name to the corresponding linear layer.

As we saw earlier (in Assignment 2 and elsewhere), a Transformer model consists of a hierarchy of nested submodules. Each of these can be addressed by a fully-qualified string name. You can use get_submodule() to retrieve a layer by a string name. This name depends on the model you have selected. For instance, in the `SmolLM2-135M` model, `'model.layers.0.self_attn.q_proj'`
 refers to the query projection in Transformer layer 0.

It is OK to hard-code this part, so that you just enumerate the layers you want to extract. Alternatively, use a utility such as `model.named_modules()` to iterate through the model's layers.

In [26]:
import torch.nn as nn

def extract_lora_targets(model):
    """
    Extract specific linear layers from the model to be wrapped with LoRA logic.
    """
    targets = {}
    # Iterate through all named submodules recursively
    for name, module in model.named_modules():
        # Check if the module is a linear layer
        if isinstance(module, nn.Linear):
            # Target standard attention projection matrices
            if any(proj in name for proj in ['q_proj', 'k_proj', 'v_proj', 'o_proj']):
                targets[name] = module
    return targets

We also need a convenience function that puts layers back into a model. The following function does the trick. The `named_layers` argument uses the same format as returned by `extract_lora_targets`.

In [27]:
def replace_layers(model, named_layers):
    """
    Replace submodules in `model` by name.
    """
    for name, layer in named_layers.items():
        components = name.split(".")
        submodule = model
        for comp in components[:-1]:
            submodule = getattr(submodule, comp)
        setattr(submodule, components[-1], layer)
    return model

### 🎓&nbsp; Task 4.2: Implementing the LoRA layer

To implement the LoRA approach, we define a new type of layer that will be used as a drop-in replacement for a regular linear layer.

In [the paper by Hu et al. (2021)](https://arxiv.org/pdf/2106.09685), the structure is presented visually in Figure 1, and equation (3) shows the same idea.

Start from the following skeleton and fill in the missing pieces:


In [28]:
class LoRALayer(nn.Module):
    """
    LoRA Mathematical Intuition:
    Instead of updating a massive weight matrix W via W_new = W + ΔW,
    LoRA assumes that the update matrix ΔW has a very low "intrinsic rank".
    Therefore, we can approximate ΔW by multiplying two much smaller matrices:
    ΔW = B * A.
    This drops the trainable parameters from (Dim x Dim) down to (Dim x r) + (r x Dim).
    """
    def __init__(self, W, r, alpha):
        super().__init__()
        self.W = W
        self.r = r           # The 'bottleneck' rank (usually 8, 16, or 32)
        self.alpha = alpha   # The scaling factor constant
        self.scaling = self.alpha / self.r # Normalizes gradients when experimenting with different r values

        # Matrix A (Down-projection): maps from original input features down to tiny rank 'r'
        self.lora_A = nn.Linear(W.in_features, r, bias=False)
        # Matrix B (Up-projection): maps from tiny rank 'r' back to original output features
        self.lora_B = nn.Linear(r, W.out_features, bias=False)

        # Initialization is critical:
        # A is random. B is EXACTLY zero.
        # Because B is zero, (B * A) = 0 at step 0.
        # This guarantees that our model starts off completely unchanged from the pre-trained state.
        nn.init.normal_(self.lora_A.weight, std=0.02)
        nn.init.zeros_(self.lora_B.weight)

        # Memory Optimization: Turn off gradients for the gigantic original base weights.
        # They are "frozen" and act only as a static background computation.
        self.W.weight.requires_grad = False
        if getattr(self.W, "bias", None) is not None:
            self.W.bias.requires_grad = False

    def forward(self, x):
        # 1. Compute the original output using frozen weights
        base_out = self.W(x)
        # 2. Compute the delta (ΔW) through the low-rank bottleneck path and scale it
        lora_out = self.lora_B(self.lora_A(x)) * self.scaling
        # 3. Add them together. The network "adapts" the frozen knowledge with tiny flexible adjustments.
        return base_out + lora_out

Here, `W` is the linear layer we are fine-tuning, while `r` and `alpha` are hyperparameters described in section 4.1. of the paper. The `r` parameter controls the parameter efficiency: by setting it to a low value, we save memory but make a rougher approximation. The `alpha` parameter is a scaling factor.

### 🎓&nbsp; Task 4.3: Fine-tuning with LoRA

Set up a model where you replace the four linear layers in attention blocks (query, key, value, and output) with LoRA layers. Use the following steps:
- First use `extract_lora_targets` to get the relevant linear layers.
- Each of the linear layers in the returned dictionary should be wrapped inside a LoRA layer.
- Then use `replace_layers` to put them back into the model.

Train this model and compare the training speed, metrics, and outputs to the results from Part 3.

Apply your parameter counting function (`num_trainable_parameters`) to this model, compare the results to those in Part 3, and make sure that these results correspond to your expectations.


In [29]:
print("\n" + "=" * 80)
print("PREPARING LORA MODEL")
print("=" * 80)

# Load a clean pre-trained model for LoRA adaptation
lora_base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(DEVICE)
targets = extract_lora_targets(lora_base_model)

# Wrap the targeted layers (commonly r=8 and alpha=16 provides a solid baseline)
lora_layers = {name: LoRALayer(layer, r=8, alpha=16) for name, layer in targets.items()}
lora_model = replace_layers(lora_base_model, lora_layers)

print(f"Total parameters: {sum(p.numel() for p in lora_model.parameters())}")
print(f"Trainable parameters (LoRA): {num_trainable_parameters(lora_model)}")

lora_trainer = make_trainer(lora_model, baseline_training_args)

print("\n" + "=" * 80)
print("TRAINING LORA MODEL")
print("=" * 80)
lora_trainer.train()

print("\n" + "=" * 80)
print("EVALUATING LORA MODEL")
print("=" * 80)
lora_eval_metrics = lora_trainer.evaluate()
print(json.dumps(lora_eval_metrics, indent=2))


PREPARING LORA MODEL


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Total parameters: 135436608
Trainable parameters (LoRA): 108894528

TRAINING LORA MODEL


  0%|          | 0/5000 [00:00<?, ?it/s]

{'loss': '1.611', 'grad_norm': '4.187', 'learning_rate': '3.001e-05', 'epoch': '0.4'}
{'loss': '1.369', 'grad_norm': '8.059', 'learning_rate': '1.001e-05', 'epoch': '0.8'}


  0%|          | 0/400 [00:00<?, ?it/s]

{'eval_loss': '1.389', 'eval_rougeL': '0.6402', 'eval_runtime': '44.12', 'eval_samples_per_second': '9.067', 'eval_steps_per_second': '9.067', 'epoch': '1'}
{'train_runtime': '1093', 'train_samples_per_second': '4.573', 'train_steps_per_second': '4.573', 'train_loss': '1.452', 'epoch': '1'}

EVALUATING LORA MODEL


  0%|          | 0/400 [00:00<?, ?it/s]

{
  "eval_loss": 1.3892922401428223,
  "eval_rougeL": 0.6402472925098606,
  "eval_runtime": 43.6834,
  "eval_samples_per_second": 9.157,
  "eval_steps_per_second": 9.157,
  "epoch": 1.0
}


### 🎓&nbsp; Task 4.4: Qualitative inspection

Run the three models interactively on some examples of your own choice (either taken from the training or test sets, or created by yourself). The convenience function below can be of use, but you need to complete it by using the prompt format you defined in Task 1.2.

Do your models seem to have learned the instruction-following behavior (at least to some extent)? Do they respond to user queries sensibly?

The quality we see here will depend on your choice of base model as well as how much you trained it.

In [30]:
def generate_inference(model, tokenizer, query):
    """
    Performs live inference (text generation) simulating a real chat interface.
    It strictly wraps the user's string into the exact ChatML format the model was trained on.
    """
    # Step 1: Prime the model with the exact formatting syntax learned during SFT
    prompt = f"<|im_start|>user\n{query}<|im_end|>\n<|im_start|>assistant\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)

    # Step 2: Grab the integer ID representing our stopping token '<|im_end|>'
    eos_token_id = tokenizer.vocab.get('<|im_end|>', tokenizer.eos_token_id)

    # Step 3: Autoregressive Generation
    # no_grad() prevents PyTorch from storing activation history, saving VRAM during inference.
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=64, # Limit the output length
            eos_token_id=eos_token_id, # Crucial: Force the model to stop generating when it thinks the answer is done
            pad_token_id=tokenizer.pad_token_id
        )

    # Decode the resulting tensor array back into a human-readable English string
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Testing with an unseen prompt to check generalization capabilities
test_query = "Could you briefly explain the concept of Federated Learning and its primary security challenges?"

print("=== Base SFT Model Output ===")
print(generate_inference(base_model, tokenizer, test_query))

print("\n=== LoRA Model Output ===")
print(generate_inference(lora_model, tokenizer, test_query))

[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


=== Base SFT Model Output ===


[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


user
Could you briefly explain the concept of Federated Learning and its primary security challenges?
assistant

The concept of federated learning is a powerful tool for building deep learning models that can learn from multiple sources simultaneously. It allows us to combine the strengths of different sources, such as GPUs, CPUs, and GPUs, to achieve better performance and reduce the risk of overfitting. However, federated learning also introduces new

=== LoRA Model Output ===
user
Could you briefly explain the concept of Federated Learning and its primary security challenges?
assistant

The concept of federated learning is a way to combine the power of traditional machine learning with the security of distributed computing. By using multiple computers to process data, federated learning can provide a more robust and secure solution to complex problems.

The primary challenge with federated learning is ensuring the security of the data
